# PRAME MIL Attention Heatmaps — Colab (CPU runtime)

Renders the 48-figure curated heatmap panel (top-12 TP/TN/FP/FN by confidence) from the trained Component 1 MIL models. For each slide, attention weights come from the **held-out fold** only (no training leakage).

## Why Colab, specifically the CPU runtime

- **Bandwidth** is the reason this runs in Colab, not on a laptop. Re-downloading ~50 WSIs (~10 GB) from GDC is ~3–6× faster on Colab's datacenter fabric than a home connection (~2–4 min vs ~15–40 min).
- **Compute** is trivial: MIL inference is a few linear layers over ~30k features per slide (~60 s for all 200 slides on CPU); rendering uses a rasterized overlay (one numpy canvas + single `imshow`, ~1 s/slide).
- **GPU gives no meaningful speedup** here — the bottlenecks are network, `openslide` thumbnail decoding, and matplotlib, none of which are GPU-accelerable. T4 would only save ~1 min on the OOF inference step. L4/A100 give nothing extra.
- This notebook **pins `device = torch.device('cpu')` explicitly**, so even if a GPU runtime is accidentally attached, the job runs on CPU and **burns zero compute units**.

Expected total wall-clock on Colab CPU: **~5–8 min** for the 48-slide panel.

## What this notebook does

1. Mount Drive, clone the repo
2. Load helpers from `05_generate_heatmaps.py` via `SourceFileLoader`
3. Compute (or reuse cached) out-of-fold predictions for all 200 slides
4. Select the top-N per TP/TN/FP/FN class by `|logit|` (default N=12 → 48 slides)
5. Download those WSIs **in parallel** and render each as it lands, deleting `.svs` immediately to cap `/content` disk
6. Save PNGs to Drive and display inline

## 1. Setup — system + Python deps

In [ ]:
!apt-get -qq install -y openslide-tools
!pip install -q openslide-python h5py

## 2. Mount Drive and clone / refresh the repo

In [ ]:
from pathlib import Path
import subprocess
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/hb-1968/prame-predict.git'
REPO_ROOT = Path('/content/prame-predict')

if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=False)

print('repo ready at', REPO_ROOT)

## 3. Config

All knobs live here. Edit and re-run from this cell downward.

In [ ]:
MODEL = 'uni'                 # 'uni' or 'conch'
N_PER_CLASS = 12              # 4 x 12 = 48 slides (~50-slide panel)
SINGLE_SLIDE = None           # set to a stem to render just one slide
FORCE_REFRESH = False         # re-render PNGs that already exist on Drive
DOWNLOAD_WORKERS = 8          # parallel GDC streams
THUMBNAIL_MAX_PX = 4096
ALPHA = 0.5
CMAP = 'viridis'

DRIVE_ROOT  = Path('/content/drive/MyDrive/prame-predict')
EMB_DIR     = DRIVE_ROOT / 'embeddings' / MODEL
RESULTS_DIR = DRIVE_ROOT / 'results' / MODEL
OUT_DIR     = RESULTS_DIR / 'heatmaps'
WSI_TMP     = Path('/content/wsi_batch')
MANIFEST    = REPO_ROOT / 'data/expression/slide_manifest.csv'

OUT_DIR.mkdir(parents=True, exist_ok=True)
WSI_TMP.mkdir(parents=True, exist_ok=True)

FEAT_DIM = {'uni': 1024, 'conch': 768}[MODEL]

print(f'embeddings : {EMB_DIR}')
print(f'results    : {RESULTS_DIR}')
print(f'heatmaps   : {OUT_DIR}')

## 4. Load helpers from 05_generate_heatmaps.py

No logic duplication — the notebook uses the same functions the CLI script uses. The script’s `_load_sibling` helper loads `02_tile_wsi.py` and `04_train_mil.py` via `importlib` (numeric-prefix filenames aren’t valid module names).

In [ ]:
from importlib.machinery import SourceFileLoader

hm = SourceFileLoader('hm', str(REPO_ROOT / '05_generate_heatmaps.py')).load_module()
train_mil = hm._load_sibling('train_mil_helpers', '04_train_mil.py')
tile_wsi  = hm._load_sibling('tile_wsi_helpers',  '02_tile_wsi.py')
AttentionMIL  = train_mil.AttentionMIL
get_20x_level = tile_wsi.get_20x_level

print('helpers loaded')

## 5. Device, manifest, and CV fold replay

Device is **pinned to CPU** on purpose — no runtime checks, no `cuda.is_available()` fallback. This guarantees zero compute units are consumed even if a GPU runtime is attached by accident. See the intro for the reasoning.

In [ ]:
import torch
import pandas as pd

device = torch.device('cpu')  # pinned; see intro for rationale
torch.set_num_threads(max(1, (torch.get_num_threads() or 2)))
print(f'device: {device}  (torch threads: {torch.get_num_threads()})')
if torch.cuda.is_available():
    print('  note: a GPU is attached but will not be used — this saves compute units.')

manifest = pd.read_csv(MANIFEST)
fold_map, *_ = hm.build_cv_fold_map(manifest, EMB_DIR)
print(f'fold_map covers {len(fold_map)} slides (expected 200)')

## 6. OOF predictions — cached on Drive

Reads all 200 `.h5` embeddings directly from Drive; no WSI traffic yet. Result is a DataFrame with one row per slide including the held-out logit, prob, and TP/TN/FP/FN outcome. Cached so later config tweaks (e.g. different `N_PER_CLASS`) reuse it.

In [ ]:
OOF_CSV = OUT_DIR / 'oof_predictions.csv'

if OOF_CSV.exists() and not FORCE_REFRESH:
    oof_df = pd.read_csv(OOF_CSV)
    print(f'cache hit: {OOF_CSV} ({len(oof_df)} rows)')
else:
    print('computing OOF predictions across 5 folds (CPU)...')
    oof_df = hm.compute_oof_predictions(
        manifest, fold_map, EMB_DIR, RESULTS_DIR,
        FEAT_DIM, AttentionMIL, device,
    )
    oof_df.to_csv(OOF_CSV, index=False)
    print(f'saved {OOF_CSV}')

from IPython.display import display
display(oof_df['outcome'].value_counts().rename('count').to_frame())

## 7. Curated selection

Picks the top-N per outcome class by `|logit|`. Set `SINGLE_SLIDE` in the config cell to a specific stem to render just one slide instead.

In [ ]:
if SINGLE_SLIDE:
    target_stem = SINGLE_SLIDE[:-4] if SINGLE_SLIDE.endswith('.svs') else SINGLE_SLIDE
    match = oof_df[oof_df['stem'] == target_stem]
    if match.empty:
        raise SystemExit(f'{SINGLE_SLIDE!r} not found in OOF predictions')
    curated = match.reset_index(drop=True)
else:
    curated = hm.select_curated_slides(oof_df, n_per_class=N_PER_CLASS).reset_index(drop=True)

print(f'rendering {len(curated)} slides')
display(curated[['outcome', 'label', 'prob', 'fold', 'stem']])

## 8. Pipelined download + render (all on Colab CPU)

Producer/consumer: `DOWNLOAD_WORKERS` threads stream `.svs` files from GDC concurrently. The main thread drains the as-completed queue and renders each slide the moment its download lands, then deletes the `.svs` immediately. Every step — download, inference, matplotlib render — runs on this Colab instance, not locally.

Already-rendered PNGs on Drive are skipped unless `FORCE_REFRESH = True`.

In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

session = requests.Session()
adapter = requests.adapters.HTTPAdapter(
    pool_connections=DOWNLOAD_WORKERS, pool_maxsize=DOWNLOAD_WORKERS,
)
session.mount('https://', adapter)

def _png_path(row):
    return OUT_DIR / f"{row['outcome']}_{row['stem']}.png"

pending = [row for _, row in curated.iterrows()
           if FORCE_REFRESH or not _png_path(row).exists()]
print(f'{len(pending)} to render, {len(curated) - len(pending)} already on Drive')

futures = {}
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as pool:
    for row in pending:
        svs_path = WSI_TMP / (row['stem'] + '.svs')
        futures[pool.submit(hm.download_wsi, row['file_id'], svs_path, session)] = row

    for fut in as_completed(futures):
        row = futures[fut]
        try:
            svs_path = fut.result()
        except Exception as exc:
            print(f"  [skip] download failed for {row['stem']}: {exc}")
            continue

        weights_path = RESULTS_DIR / f"fold{int(row['fold']) + 1}_model.pt"
        model = hm.load_fold_model(weights_path, FEAT_DIM, AttentionMIL, device)
        coords, attention, _ = hm.compute_attention(
            EMB_DIR / (row['stem'] + '.h5'), model, device,
        )
        out_png = _png_path(row)
        try:
            hm.render_heatmap(
                svs_path, coords, attention, out_png,
                meta={
                    'stem':    row['stem'],
                    'label':   int(row['label']),
                    'prob':    float(row['prob']),
                    'outcome': row['outcome'],
                    'fold':    int(row['fold']),
                },
                get_20x_level=get_20x_level,
                thumbnail_max_px=THUMBNAIL_MAX_PX,
                alpha=ALPHA,
                cmap_name=CMAP,
            )
            print(f'  wrote {out_png.name}')
        finally:
            try:
                svs_path.unlink()
            except OSError:
                pass
            del model

print('\nAll renders complete.')

## 9. Display the heatmap panel inline

In [ ]:
from IPython.display import Image, display

pngs = sorted(OUT_DIR.glob('*.png'))
print(f'{len(pngs)} heatmap(s) in {OUT_DIR}')
for png in pngs:
    print(png.name)
    display(Image(filename=str(png), width=900))

## 10. Cleanup

Remove the ephemeral `/content/wsi_batch/` directory. PNGs stay on Drive.

In [ ]:
import shutil
shutil.rmtree(WSI_TMP, ignore_errors=True)
print(f'{WSI_TMP} removed')
print(f'heatmaps live on Drive at: {OUT_DIR}')